# Module 4b: Spark vs Ray Comparison

**DSC 232R - Big Data Analysis Using Spark**

This notebook compares Spark and Ray:
1. API and programming model differences
2. Performance characteristics
3. Use case suitability
4. When to use each framework

## Key Takeaways

- **Spark** excels at ETL, SQL queries, and batch processing
- **Ray** excels at ML training, custom tasks, and stateful computation
- Choose based on workload: not "Spark OR Ray" but "Spark AND Ray"

In [ ]:
!pip install ray

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 MB 11.1 MB/s eta 0:00:00


In [ ]:
import ray
import numpy as np
import pandas as pd
import time
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Initialize both frameworks
if ray.is_initialized():
    ray.shutdown()
ray.init(num_cpus=4, logging_level="WARNING")

spark = SparkSession.builder \
    .appName("SparkRayComparison") \
    .master("local[4]") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

print(f"Ray version: {ray.__version__}")
print(f"Spark version: {spark.version}")

/usr/local/lib/python3.12/dist-packages/ray/_private/worker.py:2051: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(


Ray version: 2.55.1
Spark version: 4.0.2


---

## 1. API Comparison

### Creating Distributed Data

In [ ]:
# Generate sample data
np.random.seed(42)
n_rows = 100000

data = pd.DataFrame({
    "id": range(n_rows),
    "value": np.random.randn(n_rows),
    "category": np.random.choice(["A", "B", "C"], n_rows),
    "timestamp": pd.date_range("2023-01-01", periods=n_rows, freq="s")
})

print(f"Sample data: {len(data):,} rows")

Sample data: 100,000 rows


In [ ]:
# SPARK: Create DataFrame
spark_df = spark.createDataFrame(data)
print("Spark DataFrame:")
spark_df.show(3)
print(f"Partitions: {spark_df.rdd.getNumPartitions()}")

Spark DataFrame:
+---+--------------------+--------+-------------------+
| id|               value|category|          timestamp|
+---+--------------------+--------+-------------------+
|  0|  0.4967141530112327|       B|2023-01-01 00:00:00|
|  1|-0.13826430117118466|       B|2023-01-01 00:00:01|
|  2|  0.6476885381006925|       A|2023-01-01 00:00:02|
+---+--------------------+--------+-------------------+
only showing top 3 rows
Partitions: 4


In [ ]:
# RAY: Create Dataset
ray_ds = ray.data.from_pandas(data)
print("Ray Dataset:")
print(ray_ds)
print(f"\nSample:")
ray_ds.take(3)

2026-05-19 17:30:40,289	INFO logging.py:416 -- Registered dataset logger for dataset dataset_0_0
2026-05-19 17:30:40,312	WARNING resource_manager.py:169 -- ⚠️  Ray's object store is configured to use only 42.9% of available memory (3.7GiB out of 8.6GiB total). For optimal Ray Data performance, we recommend setting the object store to at least 50% of available memory. You can do this by setting the 'object_store_memory' parameter when calling ray.init() or by setting the RAY_DEFAULT_OBJECT_STORE_MEMORY_PROPORTION environment variable.
2026-05-19 17:30:40,316	INFO __init__.py:56 -- Progress will be logged because stdout is a non-interactive terminal.


Ray Dataset:


2026-05-19 17:30:45,262	INFO logging_progress.py:174 -- ======= Running Dataset: dataset_0_0 =======
2026-05-19 17:30:45,265	INFO logging_progress.py:225 -- Total Progress: 0/?
2026-05-19 17:30:45,266	INFO logging_progress.py:227 -- Active & requested resources: 0/4 CPU, 0.0B/1.8GiB object store
2026-05-19 17:30:45,266	INFO logging_progress.py:192 -- ============================================
2026-05-19 17:30:45,282	INFO streaming_executor.py:294 -- ✔️  Dataset dataset_0_0 execution finished in 0.00 seconds
2026-05-19 17:30:45,340	INFO dataset.py:3818 -- Tip: Use `take_batch()` instead of `take() / show()` to return records in pandas or numpy batch format.
2026-05-19 17:30:45,344	INFO logging.py:416 -- Registered dataset logger for dataset dataset_1_0
2026-05-19 17:30:45,348	INFO streaming_executor.py:166 -- Starting execution of Dataset dataset_1_0. Full logs are in /tmp/ray/session_2026-05-19_17-29-40_942208_3593/logs/ray-data
2026-05-19 17:30:45,349	INFO streaming_executor.py:167 

shape: (100000, 4)
╭───────┬──────────────────────┬──────────┬─────────────────────╮
│ id    ┆ value                ┆ category ┆ timestamp           │
│ ---   ┆ ---                  ┆ ---      ┆ ---                 │
│ int64 ┆ double               ┆ object   ┆ timestamp[ns]       │
╞═══════╪══════════════════════╪══════════╪═════════════════════╡
│ 0     ┆ 0.4967141530112327   ┆ B        ┆ 2023-01-01 00:00:00 │
│ 1     ┆ -0.13826430117118466 ┆ B        ┆ 2023-01-01 00:00:01 │
│ 2     ┆ 0.6476885381006925   ┆ A        ┆ 2023-01-01 00:00:02 │
│ 3     ┆ 1.5230298564080254   ┆ B        ┆ 2023-01-01 00:00:03 │
│ 4     ┆ -0.23415337472333597 ┆ C        ┆ 2023-01-01 00:00:04 │
│ …     ┆ …                    ┆ …        ┆ …                   │
│ 99995 ┆ -0.22522493452090622 ┆ A        ┆ 2023-01-02 03:46:35 │
│ 99996 ┆ -0.5697775451218469  ┆ C        ┆ 2023-01-02 03:46:36 │
│ 99997 ┆ 0.40918507832743084  ┆ C        ┆ 2023-01-02 03:46:37 │
│ 99998 ┆ -0.2110916707698994  ┆ B        ┆ 2023-01-02 03

[{'id': 0,
  'value': 0.4967141530112327,
  'category': 'B',
  'timestamp': Timestamp('2023-01-01 00:00:00')},
 {'id': 1,
  'value': -0.13826430117118466,
  'category': 'B',
  'timestamp': Timestamp('2023-01-01 00:00:01')},
 {'id': 2,
  'value': 0.6476885381006925,
  'category': 'A',
  'timestamp': Timestamp('2023-01-01 00:00:02')}]

### Transformations

In [ ]:
# SPARK: Transformations
spark_result = spark_df \
    .filter(F.col("value") > 0) \
    .withColumn("value_squared", F.col("value") ** 2) \
    .groupBy("category") \
    .agg(
        F.count("*").alias("count"),
        F.avg("value").alias("avg_value"),
        F.avg("value_squared").alias("avg_squared")
    )

print("Spark aggregation:")
spark_result.show()

Spark aggregation:
+--------+-----+------------------+------------------+
|category|count|         avg_value|       avg_squared|
+--------+-----+------------------+------------------+
|       B|16684|0.8034897150629284|1.0084580115530097|
|       C|16704|0.7937236981211047|0.9981026739755312|
|       A|16724|0.7966226067740564|0.9965416221578042|
+--------+-----+------------------+------------------+



In [ ]:
# RAY: Transformations
def transform_batch(batch: pd.DataFrame) -> pd.DataFrame:
    filtered = batch[batch["value"] > 0].copy()
    filtered["value_squared"] = filtered["value"] ** 2
    return filtered

ray_transformed = ray_ds.map_batches(transform_batch, batch_format="pandas")

# Aggregation
ray_agg = ray_transformed.groupby("category").mean(["value", "value_squared"])

print("Ray aggregation:")
ray_agg.take_all()

2026-05-19 17:31:27,085	INFO logging.py:416 -- Registered dataset logger for dataset dataset_3_0
2026-05-19 17:31:27,101	INFO streaming_executor.py:166 -- Starting execution of Dataset dataset_3_0. Full logs are in /tmp/ray/session_2026-05-19_17-29-40_942208_3593/logs/ray-data
2026-05-19 17:31:27,103	INFO streaming_executor.py:167 -- Execution plan of Dataset dataset_3_0: InputDataBuffer[Input] -> TaskPoolMapOperator[MapBatches(transform_batch)] -> HashAggregateOperator[HashAggregate(key_columns=('category',), num_partitions=1)]
2026-05-19 17:31:27,235	INFO logging_progress.py:174 -- ======= Running Dataset: dataset_3_0 =======
2026-05-19 17:31:27,245	INFO logging_progress.py:225 -- Total Progress: 0/?
2026-05-19 17:31:27,251	INFO logging_progress.py:227 -- Active & requested resources: 0.25/4 CPU, 0.0B/1.8GiB object store
2026-05-19 17:31:27,255	INFO logging_progress.py:181 -- 
2026-05-19 17:31:27,258	INFO logging_progress.py:231 -- MapBatches(transform_batch): 0/1
2026-05-19 17:31:27

Ray aggregation:


2026-05-19 17:31:32,738	INFO streaming_executor.py:294 -- ✔️  Dataset dataset_3_0 execution finished in 5.64 seconds


[{'category': 'A',
  'mean(value)': 0.7966226067740569,
  'mean(value_squared)': 0.9965416221578033},
 {'category': 'B',
  'mean(value)': 0.8034897150629285,
  'mean(value_squared)': 1.0084580115530086},
 {'category': 'C',
  'mean(value)': 0.7937236981211043,
  'mean(value_squared)': 0.9981026739755317}]

---

## 2. Performance Comparison

In [ ]:
# Larger dataset for meaningful comparison
n_large = 1_000_000
large_data = pd.DataFrame({
    "x": np.random.randn(n_large),
    "y": np.random.randn(n_large),
    "z": np.random.randn(n_large),
})

print(f"Large dataset: {len(large_data):,} rows, {large_data.memory_usage().sum() / 1e6:.1f} MB")

Large dataset: 1,000,000 rows, 24.0 MB


In [ ]:
# Test 1: Simple map operation
def benchmark_map():
    results = {}

    # Spark
    spark_large = spark.createDataFrame(large_data)
    start = time.time()
    spark_mapped = spark_large.withColumn("result", F.col("x") * 2 + F.col("y"))
    _ = spark_mapped.count()  # Force evaluation
    results["Spark"] = time.time() - start

    # Ray
    ray_large = ray.data.from_pandas(large_data)
    start = time.time()
    def map_func(batch):
        batch["result"] = batch["x"] * 2 + batch["y"]
        return batch
    ray_mapped = ray_large.map_batches(map_func, batch_format="pandas")
    _ = ray_mapped.count()  # Force evaluation
    results["Ray"] = time.time() - start

    return results

print("Benchmark: Map Operation")
print("="*40)
map_results = benchmark_map()
for framework, t in map_results.items():
    print(f"{framework}: {t:.3f}s")

Benchmark: Map Operation


2026-05-19 17:32:01,640	INFO logging.py:416 -- Registered dataset logger for dataset dataset_6_0
2026-05-19 17:32:01,651	INFO streaming_executor.py:166 -- Starting execution of Dataset dataset_6_0. Full logs are in /tmp/ray/session_2026-05-19_17-29-40_942208_3593/logs/ray-data
2026-05-19 17:32:01,654	INFO streaming_executor.py:167 -- Execution plan of Dataset dataset_6_0: InputDataBuffer[Input] -> TaskPoolMapOperator[MapBatches(map_func)->Project] -> AggregateNumRows[AggregateNumRows]
2026-05-19 17:32:01,703	INFO logging_progress.py:174 -- ======= Running Dataset: dataset_6_0 =======
2026-05-19 17:32:01,710	INFO logging_progress.py:225 -- Total Progress: 0/?
2026-05-19 17:32:01,716	INFO logging_progress.py:227 -- Active & requested resources: 0/4 CPU, 0.0B/1.8GiB object store
2026-05-19 17:32:01,717	INFO logging_progress.py:181 -- 
2026-05-19 17:32:01,718	INFO logging_progress.py:231 -- MapBatches(map_func)->Project: 0/1
2026-05-19 17:32:01,719	INFO logging_progress.py:233 --   Tasks: 

Spark: 5.388s
Ray: 0.260s


In [ ]:
# Test 2: Aggregation
def benchmark_aggregation():
    results = {}

    # Create categorical data for groupby
    agg_data = large_data.copy()
    agg_data["group"] = np.random.choice(["A", "B", "C", "D", "E"], len(agg_data))

    # Spark
    spark_agg = spark.createDataFrame(agg_data)
    start = time.time()
    result = spark_agg.groupBy("group").agg(
        F.avg("x").alias("avg_x"),
        F.sum("y").alias("sum_y"),
        F.count("*").alias("count")
    )
    _ = result.collect()
    results["Spark"] = time.time() - start

    # Ray
    ray_agg = ray.data.from_pandas(agg_data)
    start = time.time()
    result = ray_agg.groupby("group").mean(["x", "y"])
    _ = result.take_all()
    results["Ray"] = time.time() - start

    return results

print("\nBenchmark: Aggregation")
print("="*40)
agg_results = benchmark_aggregation()
for framework, t in agg_results.items():
    print(f"{framework}: {t:.3f}s")


Benchmark: Aggregation


2026-05-19 17:32:34,840	INFO logging.py:416 -- Registered dataset logger for dataset dataset_8_0
2026-05-19 17:32:34,848	INFO hash_aggregate.py:161 -- Estimated memory requirement for aggregating aggregator (partitions=1, aggregators=1, dataset (estimate)=0.1GiB): shuffle=70.6MiB, output=70.6MiB, total=141.1MiB, 
2026-05-19 17:32:34,854	INFO streaming_executor.py:166 -- Starting execution of Dataset dataset_8_0. Full logs are in /tmp/ray/session_2026-05-19_17-29-40_942208_3593/logs/ray-data
2026-05-19 17:32:34,856	INFO streaming_executor.py:167 -- Execution plan of Dataset dataset_8_0: InputDataBuffer[Input] -> HashAggregateOperator[HashAggregate(key_columns=('group',), num_partitions=1)]
2026-05-19 17:32:34,907	INFO logging_progress.py:174 -- ======= Running Dataset: dataset_8_0 =======
2026-05-19 17:32:34,910	INFO logging_progress.py:225 -- Total Progress: 0/?
2026-05-19 17:32:34,917	INFO logging_progress.py:227 -- Active & requested resources: 0.03/4 CPU, 0.0B/1.8GiB object store
20

Spark: 4.278s
Ray: 1.887s


In [ ]:
# Test 3: Custom Python function (Ray's strength)
def benchmark_custom_udf():
    results = {}

    # Spark with UDF (slow due to serialization)
    from pyspark.sql.functions import udf
    from pyspark.sql.types import DoubleType

    @udf(DoubleType())
    def custom_func_spark(x, y, z):
        return float(np.sin(x) + np.cos(y) + np.sqrt(abs(z)))

    spark_udf = spark.createDataFrame(large_data)
    start = time.time()
    result = spark_udf.withColumn("custom", custom_func_spark("x", "y", "z"))
    _ = result.count()
    results["Spark UDF"] = time.time() - start

    # Ray with vectorized function (fast)
    def custom_func_ray(batch: pd.DataFrame) -> pd.DataFrame:
        batch["custom"] = np.sin(batch["x"]) + np.cos(batch["y"]) + np.sqrt(np.abs(batch["z"]))
        return batch

    ray_udf = ray.data.from_pandas(large_data)
    start = time.time()
    result = ray_udf.map_batches(custom_func_ray, batch_format="pandas")
    _ = result.count()
    results["Ray vectorized"] = time.time() - start

    return results

print("\nBenchmark: Custom Python Function")
print("="*40)
udf_results = benchmark_custom_udf()
for framework, t in udf_results.items():
    print(f"{framework}: {t:.3f}s")
print(f"\nSpeedup: {udf_results['Spark UDF']/udf_results['Ray vectorized']:.1f}x")


Benchmark: Custom Python Function


2026-05-19 17:33:03,260	INFO logging.py:416 -- Registered dataset logger for dataset dataset_11_0
2026-05-19 17:33:03,269	INFO streaming_executor.py:166 -- Starting execution of Dataset dataset_11_0. Full logs are in /tmp/ray/session_2026-05-19_17-29-40_942208_3593/logs/ray-data
2026-05-19 17:33:03,270	INFO streaming_executor.py:167 -- Execution plan of Dataset dataset_11_0: InputDataBuffer[Input] -> TaskPoolMapOperator[MapBatches(custom_func_ray)->Project] -> AggregateNumRows[AggregateNumRows]
2026-05-19 17:33:03,319	INFO logging_progress.py:174 -- ======= Running Dataset: dataset_11_0 =======
2026-05-19 17:33:03,324	INFO logging_progress.py:225 -- Total Progress: 0/?
2026-05-19 17:33:03,325	INFO logging_progress.py:227 -- Active & requested resources: 0/4 CPU, 0.0B/1.8GiB object store
2026-05-19 17:33:03,327	INFO logging_progress.py:181 -- 
2026-05-19 17:33:03,333	INFO logging_progress.py:231 -- MapBatches(custom_func_ray)->Project: 0/1
2026-05-19 17:33:03,336	INFO logging_progress.p

Spark UDF: 2.175s
Ray vectorized: 0.232s

Speedup: 9.4x


---

## 3. Feature Comparison

In [ ]:
comparison_table = pd.DataFrame({
    "Feature": [
        "Primary Use Case",
        "Programming Model",
        "SQL Support",
        "UDF Performance",
        "Stateful Computation",
        "ML Training",
        "Streaming",
        "GPU Support",
        "Fault Tolerance",
        "Language Support"
    ],
    "Spark": [
        "ETL, Data Processing",
        "DataFrame/RDD",
        "Excellent (Spark SQL)",
        "Slow (serialization)",
        "Limited",
        "MLlib (basic)",
        "Structured Streaming",
        "Rapids (limited)",
        "RDD lineage",
        "Scala, Python, Java, R"
    ],
    "Ray": [
        "ML, Custom Tasks",
        "Tasks/Actors",
        "Basic",
        "Fast (native Python)",
        "Actors",
        "Ray Train (excellent)",
        "Basic",
        "Native",
        "Object reconstruction",
        "Python (primary)"
    ]
})

print("Feature Comparison: Spark vs Ray")
print("="*80)
print(comparison_table.to_string(index=False))

Feature Comparison: Spark vs Ray
             Feature                  Spark                   Ray
    Primary Use Case   ETL, Data Processing      ML, Custom Tasks
   Programming Model          DataFrame/RDD          Tasks/Actors
         SQL Support  Excellent (Spark SQL)                 Basic
     UDF Performance   Slow (serialization)  Fast (native Python)
Stateful Computation                Limited                Actors
         ML Training          MLlib (basic) Ray Train (excellent)
           Streaming   Structured Streaming                 Basic
         GPU Support       Rapids (limited)                Native
     Fault Tolerance            RDD lineage Object reconstruction
    Language Support Scala, Python, Java, R      Python (primary)


---

## 4. When to Use Each Framework

In [ ]:
use_cases = {
    "USE SPARK WHEN": [
        "Complex SQL queries and joins",
        "ETL pipelines with structured data",
        "Data warehousing operations",
        "Integrating with Hive/HDFS ecosystem",
        "Need for data lineage and governance",
        "Heavy shuffle operations",
        "Existing Spark infrastructure"
    ],
    "USE RAY WHEN": [
        "ML model training (XGBoost, PyTorch, TF)",
        "Custom Python algorithms",
        "Stateful computation (actors)",
        "Real-time inference",
        "Reinforcement learning",
        "Hyperparameter tuning",
        "Need native Python performance"
    ],
    "USE BOTH WHEN": [
        "ETL with Spark → ML with Ray",
        "Complex data prep + model training",
        "Data validation (Spark) + Feature eng (Ray)",
        "Need best of both worlds"
    ]
}

print("Framework Selection Guide")
print("="*60)
for category, items in use_cases.items():
    print(f"\n{category}:")
    for item in items:
        print(f"  • {item}")

Framework Selection Guide

USE SPARK WHEN:
  • Complex SQL queries and joins
  • ETL pipelines with structured data
  • Data warehousing operations
  • Integrating with Hive/HDFS ecosystem
  • Need for data lineage and governance
  • Heavy shuffle operations
  • Existing Spark infrastructure

USE RAY WHEN:
  • ML model training (XGBoost, PyTorch, TF)
  • Custom Python algorithms
  • Stateful computation (actors)
  • Real-time inference
  • Reinforcement learning
  • Hyperparameter tuning
  • Need native Python performance

USE BOTH WHEN:
  • ETL with Spark → ML with Ray
  • Complex data prep + model training
  • Data validation (Spark) + Feature eng (Ray)
  • Need best of both worlds


---

## 5. Practical Decision Flowchart

In [ ]:
decision_flowchart = """
FRAMEWORK DECISION FLOWCHART
============================

Start: What's your primary task?
           │
           ├── SQL queries / Complex joins?
           │         │
           │         └── YES ──> SPARK
           │
           ├── ML Model Training?
           │         │
           │         └── YES ──> RAY
           │
           ├── Custom Python algorithms?
           │         │
           │         └── YES ──> RAY
           │
           ├── Batch ETL?
           │         │
           │         └── YES ──> SPARK
           │
           ├── Need stateful computation?
           │         │
           │         └── YES ──> RAY (Actors)
           │
           └── Both ETL and ML?
                     │
                     └── YES ──> BOTH (Spark ETL → Ray ML)
"""
print(decision_flowchart)


FRAMEWORK DECISION FLOWCHART

Start: What's your primary task?
           │
           ├── SQL queries / Complex joins?
           │         │
           │         └── YES ──> SPARK
           │
           ├── ML Model Training?
           │         │
           │         └── YES ──> RAY
           │
           ├── Custom Python algorithms?
           │         │
           │         └── YES ──> RAY
           │
           ├── Batch ETL?
           │         │
           │         └── YES ──> SPARK
           │
           ├── Need stateful computation?
           │         │
           │         └── YES ──> RAY (Actors)
           │
           └── Both ETL and ML?
                     │
                     └── YES ──> BOTH (Spark ETL → Ray ML)



---

## 6. Exercise: Choose the Right Framework

For each scenario, decide whether to use Spark, Ray, or both:

In [ ]:
scenarios = [
    {
        "scenario": "Process 500GB of web logs, aggregate by user, join with user profiles",
        "answer": "SPARK - Complex joins and aggregations on large structured data"
    },
    {
        "scenario": "Train an XGBoost model on 100GB feature dataset",
        "answer": "RAY - ML training with Ray Train XGBoostTrainer"
    },
    {
        "scenario": "Build a recommendation system that maintains user state",
        "answer": "RAY - Stateful computation with Actors"
    },
    {
        "scenario": "ETL pipeline: clean data, create features, train model, serve predictions",
        "answer": "BOTH - Spark for ETL, Ray for ML training and serving"
    },
    {
        "scenario": "Real-time fraud detection with model inference",
        "answer": "RAY - Low-latency inference with Ray Serve"
    },
]

print("Framework Selection Exercises")
print("="*70)
for i, item in enumerate(scenarios, 1):
    print(f"\n{i}. {item['scenario']}")
    print(f"   → {item['answer']}")

Framework Selection Exercises

1. Process 500GB of web logs, aggregate by user, join with user profiles
   → SPARK - Complex joins and aggregations on large structured data

2. Train an XGBoost model on 100GB feature dataset
   → RAY - ML training with Ray Train XGBoostTrainer

3. Build a recommendation system that maintains user state
   → RAY - Stateful computation with Actors

4. ETL pipeline: clean data, create features, train model, serve predictions
   → BOTH - Spark for ETL, Ray for ML training and serving

5. Real-time fraud detection with model inference
   → RAY - Low-latency inference with Ray Serve


---

## Summary

### Key Differences

| Aspect | Spark | Ray |
|--------|-------|-----|
| Strength | SQL, ETL, Joins | ML, Custom Python |
| Data Model | DataFrame/RDD | Tasks/Actors |
| State | Stateless transforms | Stateful actors |
| Python UDFs | Slow (serialization) | Fast (native) |

### Best Practice

**Don't choose one - use both!**

```
Spark ETL → Parquet → Ray ML → Model
```

### Next Steps

See `04c_data_handoff_patterns.ipynb` for data transfer strategies.

In [ ]:
# Cleanup
spark.stop()
ray.shutdown()
print("Cleanup complete.")

Cleanup complete.
